In [ ]:
!pip install gradio PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.0 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import torch
import torch.nn.functional as F
import PyPDF2
import os
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# --- CONFIGURATION ---
MODEL_PATH = "best_model-3.pth"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABELS = ["Safe Email", "Phishing Email"] # 0, 1

class PhishingDetector:
    def __init__(self, model_path):
        self.model_path = model_path
        self.device = DEVICE
        self.tokenizer = None
        self.model = None
        self._load_resources()

    def _load_resources(self):
        """Loads the tokenizer and model with error handling."""
        print(f"🔄 Initializing on {self.device}...")

        # 1. Load Tokenizer
        try:
            self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        except Exception as e:
            raise RuntimeError(f"Failed to load Tokenizer: {e}")

        # 2. Check File Exists
        if not os.path.exists(self.model_path):
            raise FileNotFoundError(f"❌ CRITICAL: '{self.model_path}' not found. Please upload it.")

        # 3. Load Model Architecture
        self.model = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased', num_labels=2
        )

        # 4. Load State Dict (Weights)
        try:
            # Try loading with weights_only=False (Newer PyTorch default safety)
            checkpoint = torch.load(self.model_path, map_location=self.device, weights_only=False)
        except TypeError:
            # Fallback for older PyTorch versions
            checkpoint = torch.load(self.model_path, map_location=self.device)

        # Handle whether checkpoint is full dict or just state_dict
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            self.model.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.model.load_state_dict(checkpoint)

        self.model.to(self.device)
        self.model.eval()
        print("✅ Model loaded successfully!")

    def predict(self, text):
        """Runs prediction on text string."""
        if not text or len(text.strip()) == 0:
            return None

        # Preprocess
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(self.device)

        # Inference
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)

        # Return dictionary for Gradio Label
        return {
            LABELS[0]: float(probs[0][0]),
            LABELS[1]: float(probs[0][1])
        }

# --- HELPER FUNCTIONS ---

def process_pdf(file_obj):
    """Extracts text from PDF and runs prediction."""
    if file_obj is None:
        return None

    try:
        # Get path (Compatible with Gradio 3.x and 4.x)
        pdf_path = file_obj.name if hasattr(file_obj, 'name') else file_obj

        reader = PyPDF2.PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            extract = page.extract_text()
            if extract:
                text += extract + "\n"

        if len(text.strip()) < 10:
            raise ValueError("PDF seems empty or unreadable.")

        return detector.predict(text)

    except Exception as e:
        return {f"Error: {str(e)}": 0.0}

# --- INITIALIZE SYSTEM ---
try:
    detector = PhishingDetector(MODEL_PATH)
except Exception as e:
    print(e)
    # create a dummy detector if model fails so UI can still launch and show error
    detector = None

def ui_predict_text(text):
    if detector is None: return {"System Error: Model not loaded": 1.0}
    return detector.predict(text)

def ui_predict_pdf(pdf):
    if detector is None: return {"System Error: Model not loaded": 1.0}
    return process_pdf(pdf)

# --- GRADIO UI (BLOCKS) ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🛡️ UofT Phishing Detector
        **Upload an email PDF or paste text to analyze for phishing risks.**
        """
    )

    with gr.Tabs():
        # TAB 1: TEXT
        with gr.TabItem("📝 Paste Text"):
            with gr.Row():
                with gr.Column():
                    text_input = gr.Textbox(lines=8, placeholder="Paste email content here...", label="Email Body")
                    text_btn = gr.Button("Analyze Text", variant="primary")
                with gr.Column():
                    text_output = gr.Label(num_top_classes=2, label="Risk Assessment")

            text_btn.click(ui_predict_text, inputs=text_input, outputs=text_output)

        # TAB 2: PDF
        with gr.TabItem("📄 Upload PDF"):
            with gr.Row():
                with gr.Column():
                    pdf_input = gr.File(label="Upload Email PDF", type="filepath")
                    pdf_btn = gr.Button("Analyze PDF", variant="primary")
                with gr.Column():
                    pdf_output = gr.Label(num_top_classes=2, label="Risk Assessment")

            pdf_btn.click(ui_predict_pdf, inputs=pdf_input, outputs=pdf_output)

    gr.Markdown("--- \n *Powered by DistilBERT & PyTorch*")

# Launch
if __name__ == "__main__":
    demo.launch(share=True, debug=True)

🔄 Initializing on cuda...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully!


/tmp/ipython-input-1570083544.py:126: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://90eaff8a8c466452e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
